# Pricify: End-to-End Price Prediction Pipeline

Notebook ini membangun pipeline lengkap dari preprocessing data Mercari, baseline models, graph construction, pelatihan GraphSAGE dan GAT, evaluasi, hingga penyimpanan artefak model ke disk.

In [1]:
# --- Section: Environment setup (Colab) ---
# Cell ini mengecek dependency yang belum terpasang, lalu menginstalnya hanya jika perlu.
import importlib.util
import subprocess
import sys
import os

is_colab = 'COLAB_RELEASE_TAG' in os.environ or 'google.colab' in sys.modules

required_packages = {
    'xgboost': 'xgboost',
    'sklearn': 'scikit-learn',
    'pandas': 'pandas',
    'numpy': 'numpy',
    'scipy': 'scipy'
}

missing_packages = []
for module_name, package_name in required_packages.items():
    if importlib.util.find_spec(module_name) is None:
        missing_packages.append(package_name)

torch_ok = False
try:
    import torch  # noqa: F401
    torch_ok = True
except Exception:
    torch_ok = False

if missing_packages:
    print('Installing missing packages:', ', '.join(missing_packages))
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q'] + missing_packages)

if not torch_ok:
    print('Torch import failed; reinstalling a compatible wheel.')
    if is_colab:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade', '--force-reinstall', 'torch', 'torchvision', 'torchaudio'])
    else:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade', '--force-reinstall', 'torch', 'torchvision', 'torchaudio', '--index-url', 'https://download.pytorch.org/whl/cpu'])
else:
    print('All required packages are already installed and torch imports correctly.')

All required packages are already installed and torch imports correctly.


In [4]:
print('Installing torch-geometric and its dependencies...')
import torch
import sys
import subprocess
import os

# Get PyTorch version without build metadata (e.g., '2.1.0' from '2.1.0+cu121')
TORCH_PACKAGE_VERSION = torch.__version__.split('+')[0]

# Assuming is_colab is defined in a previous cell, as per context.
# If not, it needs to be redefined here or made globally accessible.

if is_colab:
    # In Colab, we prefer CUDA wheels if available
    try:
        cuda_version = torch.version.cuda
        if cuda_version:
            CUDA_SUFFIX = 'cu' + cuda_version.replace('.', '')
            index_url = f"https://data.pyg.org/whl/torch-{TORCH_PACKAGE_VERSION}+{CUDA_SUFFIX}.html"
            print(f"Detected PyTorch: {TORCH_PACKAGE_VERSION}, CUDA: {cuda_version}. Attempting to install with {index_url}")
            subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
                                   'torch-scatter', 'torch-sparse', 'torch-geometric',
                                   '-f', index_url])
        else:
            # No CUDA detected, fall back to CPU for Colab
            print(f"CUDA not detected on Colab. Attempting to install CPU version for PyTorch {TORCH_PACKAGE_VERSION}.")
            subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
                                   'torch-scatter', 'torch-sparse', 'torch-geometric',
                                   '-f', f'https://data.pyg.org/whl/torch-{TORCH_PACKAGE_VERSION}+cpu.html'])
    except AttributeError:
        # If torch.version.cuda attribute doesn't exist, it's likely a CPU-only build
        print(f"torch.version.cuda not found. Attempting to install CPU version for PyTorch {TORCH_PACKAGE_VERSION}.")
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
                               'torch-scatter', 'torch-sparse', 'torch-geometric',
                               '-f', f'https://data.pyg.org/whl/torch-{TORCH_PACKAGE_VERSION}+cpu.html'])
else:
    # Not in Colab, or Colab detection failed, or explicitly CPU environment
    print(f"Not in Colab or CUDA not preferred. Attempting to install CPU version for PyTorch {TORCH_PACKAGE_VERSION}.")
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
                           'torch-scatter', 'torch-sparse', 'torch-geometric',
                           '-f', f'https://data.pyg.org/whl/torch-{TORCH_PACKAGE_VERSION}+cpu.html'])

print('torch-geometric and its dependencies installed.')

Installing torch-geometric and its dependencies...
CUDA not detected on Colab. Attempting to install CPU version for PyTorch 2.10.0.
torch-geometric and its dependencies installed.


## Preprocessing

Memuat data mentah, melakukan cleaning, membuat target log_price, serta melakukan split train/validation/test tanpa leakage.

In [6]:
# --- Section: Load and clean raw data ---
import os
import re
import pickle
import random
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import Adam
from torch.optim.lr_scheduler import ReduceLROnPlateau

from sklearn.decomposition import TruncatedSVD
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, mean_squared_log_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler

import xgboost as xgb

from torch_geometric.data import HeteroData
from torch_geometric.nn import GATConv, HeteroConv, SAGEConv
from torch_geometric.transforms import ToUndirected

SEED = 42
MAX_ROWS = 200_000
DATA_DIR = Path('/content') # Corrected path to data files
TRAIN_FILE = DATA_DIR / 'train.tsv'
ARTIFACT_DIR = Path('.')


def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


set_seed(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

df = pd.read_csv(TRAIN_FILE, sep='\t', nrows=MAX_ROWS)
df = df.copy()
df = df[df['price'] > 0].reset_index(drop=True)

df['brand_name'] = df['brand_name'].fillna('No Brand')
df['item_description'] = df['item_description'].fillna('')
df['category_name'] = df['category_name'].fillna('Other/Other/Other')


def split_category(value):
    parts = str(value).split('/')
    parts = (parts + ['Other', 'Other', 'Other'])[:3]
    return parts[0], parts[1], parts[2]


cat_parts = df['category_name'].apply(split_category)
df['cat_1'] = cat_parts.apply(lambda x: x[0])
df['cat_2'] = cat_parts.apply(lambda x: x[1])
df['cat_3'] = cat_parts.apply(lambda x: x[2])

_clean_re = re.compile(r'[^a-z0-9]+')


def clean_text(value):
    value = str(value).lower()
    value = _clean_re.sub(' ', value)
    value = re.sub(r'\s+', ' ', value).strip()
    return value


df['name_clean'] = df['name'].fillna('').apply(clean_text)
df['brand_clean'] = df['brand_name'].apply(clean_text)
df['cat1_clean'] = df['cat_1'].apply(clean_text)
df['desc_clean'] = df['item_description'].apply(clean_text)
df['text_combined'] = df['name_clean'] + ' ' + df['brand_clean'] + ' ' + df['cat1_clean'] + ' ' + df['desc_clean']
df['log_price'] = np.log1p(df['price'])

train_df, temp_df = train_test_split(df, test_size=0.30, random_state=SEED)
val_df, test_df = train_test_split(temp_df, test_size=0.50, random_state=SEED)

train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

print('Train / Val / Test:', len(train_df), len(val_df), len(test_df))
train_df[['name', 'brand_name', 'category_name', 'item_condition_id', 'shipping', 'price']].head(3)

Device: cpu
Train / Val / Test: 139925 29984 29985


,name,brand_name,category_name,item_condition_id,shipping,price
0,NWT Paw Patrol Hat,No Brand,Kids/Boys (4+)/Accessories,1,1,9.0
1,Black Holographic Trim Keyhole Rave Top,No Brand,"Women/Tops & Blouses/Tank, Cami",2,0,16.0
2,Vintage Nike Tshirt,Nike,Men/Tops/T-shirts,3,0,14.0


## Baseline Models

Bagian ini melatih TF-IDF + Ridge Regression dan XGBoost dengan fitur tabular yang ditentukan.

In [11]:
# --- Section: Baseline models ---

def fit_mapping(series):
    encoder = LabelEncoder()
    encoder.fit(series.astype(str).fillna('UNK'))
    mapping = {label: idx for idx, label in enumerate(encoder.classes_)}
    return encoder, mapping


def encode_with_unknown(series, mapping):
    unknown_index = len(mapping)
    return np.array([mapping.get(str(value), unknown_index) for value in series], dtype=np.int64)


tfidf_ridge = Pipeline([
    ('tfidf', TfidfVectorizer(max_features=50000, ngram_range=(1, 2), sublinear_tf=True)),
    ('ridge', Ridge(alpha=5.0))
])

tfidf_ridge.fit(train_df['text_combined'], train_df['log_price'])
val_pred_ridge = tfidf_ridge.predict(val_df['text_combined'])
test_pred_ridge = tfidf_ridge.predict(test_df['text_combined'])

le_brand, brand_mapping = fit_mapping(train_df['brand_name'])
le_cat_main, cat_main_mapping = fit_mapping(train_df['cat_1'])
le_cat_sub1, cat_sub1_mapping = fit_mapping(train_df['cat_2'])

X_train_xgb = np.column_stack([
    train_df['item_condition_id'].values,
    train_df['shipping'].values,
    encode_with_unknown(train_df['brand_name'], brand_mapping),
    encode_with_unknown(train_df['cat_1'], cat_main_mapping)
]).astype(np.float32)
X_val_xgb = np.column_stack([
    val_df['item_condition_id'].values,
    val_df['shipping'].values,
    encode_with_unknown(val_df['brand_name'], brand_mapping),
    encode_with_unknown(val_df['cat_1'], cat_main_mapping)
]).astype(np.float32)
X_test_xgb = np.column_stack([
    test_df['item_condition_id'].values,
    test_df['shipping'].values,
    encode_with_unknown(test_df['brand_name'], brand_mapping),
    encode_with_unknown(test_df['cat_1'], cat_main_mapping)
]).astype(np.float32)

y_train_xgb = train_df['log_price'].values
y_val_xgb = val_df['log_price'].values

xgb_model = xgb.XGBRegressor(
    n_estimators=500,
    max_depth=6,
    learning_rate=0.05,
    objective='reg:squarederror',
    random_state=SEED,
    n_jobs=-1,
    subsample=0.9,
    colsample_bytree=0.9
)

xgb_model.fit(
    X_train_xgb,
    y_train_xgb,
    eval_set=[(X_val_xgb, y_val_xgb)],
    verbose=False # Removed early stopping arguments entirely
)

val_pred_xgb = xgb_model.predict(X_val_xgb)
test_pred_xgb = xgb_model.predict(X_test_xgb)

print('TF-IDF + Ridge and XGBoost trained.')

TF-IDF + Ridge and XGBoost trained.


## Graph Construction

Graph dibangun sebagai heterogeneous graph dengan node product, brand, dan category. Node brand dan category menggunakan embedding di dalam model, bukan one-hot.

In [12]:
# --- Section: Heterogeneous graph construction ---
tfidf_vec_graph = TfidfVectorizer(max_features=50000, ngram_range=(1, 2), sublinear_tf=True)
tfidf_vec_graph.fit(train_df['text_combined'])

train_tfidf = tfidf_vec_graph.transform(train_df['text_combined'])
val_tfidf = tfidf_vec_graph.transform(val_df['text_combined'])
test_tfidf = tfidf_vec_graph.transform(test_df['text_combined'])

svd = TruncatedSVD(n_components=128, random_state=SEED)
train_tfidf_128 = svd.fit_transform(train_tfidf)
val_tfidf_128 = svd.transform(val_tfidf)
test_tfidf_128 = svd.transform(test_tfidf)


def build_product_features(df_part, tfidf_128):
    return np.hstack([
        tfidf_128,
        df_part['item_condition_id'].values.reshape(-1, 1),
        df_part['shipping'].values.reshape(-1, 1)
    ]).astype(np.float32)


prod_train = build_product_features(train_df, train_tfidf_128)
prod_val = build_product_features(val_df, val_tfidf_128)
prod_test = build_product_features(test_df, test_tfidf_128)

scaler = StandardScaler()
prod_train = scaler.fit_transform(prod_train)
prod_val = scaler.transform(prod_val)
prod_test = scaler.transform(prod_test)

prod_features = np.vstack([prod_train, prod_val, prod_test]).astype(np.float32)
full_df = pd.concat([train_df, val_df, test_df], axis=0, ignore_index=True)

brand_encoder_graph, brand_mapping_graph = fit_mapping(train_df['brand_name'])
cat_main_encoder_graph, cat_main_mapping_graph = fit_mapping(train_df['cat_1'])


def encode_graph_nodes(series, mapping):
    unknown_index = len(mapping)
    return np.array([mapping.get(str(value), unknown_index) for value in series], dtype=np.int64)


brand_ids_all = np.concatenate([
    encode_graph_nodes(train_df['brand_name'], brand_mapping_graph),
    encode_graph_nodes(val_df['brand_name'], brand_mapping_graph),
    encode_graph_nodes(test_df['brand_name'], brand_mapping_graph)
])
cat_ids_all = np.concatenate([
    encode_graph_nodes(train_df['cat_1'], cat_main_mapping_graph),
    encode_graph_nodes(val_df['cat_1'], cat_main_mapping_graph),
    encode_graph_nodes(test_df['cat_1'], cat_main_mapping_graph)
])

num_products = len(full_df)
num_brands = len(brand_mapping_graph) + 1
num_cats = len(cat_main_mapping_graph) + 1

data = HeteroData()
data['product'].x = torch.tensor(prod_features, dtype=torch.float32)
data['product'].y = torch.tensor(full_df['log_price'].values, dtype=torch.float32)
data['brand'].node_id = torch.arange(num_brands, dtype=torch.long)
data['category'].node_id = torch.arange(num_cats, dtype=torch.long)

src_index = torch.arange(num_products, dtype=torch.long)
brand_edge_index = torch.stack([
    src_index,
    torch.tensor(brand_ids_all, dtype=torch.long)
])
cat_edge_index = torch.stack([
    src_index,
    torch.tensor(cat_ids_all, dtype=torch.long)
])

data['product', 'has_brand', 'brand'].edge_index = brand_edge_index
data['product', 'in_category', 'category'].edge_index = cat_edge_index
data = ToUndirected()(data)

train_mask = torch.zeros(num_products, dtype=torch.bool)
val_mask = torch.zeros(num_products, dtype=torch.bool)
test_mask = torch.zeros(num_products, dtype=torch.bool)

train_mask[:len(train_df)] = True
val_mask[len(train_df):len(train_df) + len(val_df)] = True
test_mask[len(train_df) + len(val_df):] = True

data['product'].train_mask = train_mask
data['product'].val_mask = val_mask
data['product'].test_mask = test_mask

data = data.to(device)

in_channels_dict = {'product': data['product'].x.size(1)}
print('Product feature dimension:', in_channels_dict['product'])
print('Brands / Categories:', num_brands, num_cats)

Product feature dimension: 130
Brands / Categories: 2357 11


## Model Definitions

GraphSAGE dan GAT menggunakan embedding internal untuk brand dan category nodes, lalu memetakan node product ke output regresi.

In [20]:
class GraphSAGERegressor(nn.Module):
    def __init__(self, in_channels_dict, num_brands, num_cats, hidden_channels=128, dropout=0.3):
        super().__init__()
        self.hidden_channels = hidden_channels
        self.dropout = dropout
        self.brand_emb = nn.Embedding(num_brands, hidden_channels)
        self.cat_emb = nn.Embedding(num_cats, hidden_channels)
        self.input_proj = nn.ModuleDict({
            'product': nn.Linear(in_channels_dict['product'], hidden_channels),
            'brand': nn.Linear(hidden_channels, hidden_channels),
            'category': nn.Linear(hidden_channels, hidden_channels),
        })
        self.conv1 = HeteroConv({
            ('product', 'has_brand', 'brand'): SAGEConv((-1, -1), hidden_channels, normalize=True),
            ('brand', 'rev_has_brand', 'product'): SAGEConv((-1, -1), hidden_channels, normalize=True),
            ('product', 'in_category', 'category'): SAGEConv((-1, -1), hidden_channels, normalize=True),
            ('category', 'rev_in_category', 'product'): SAGEConv((-1, -1), hidden_channels, normalize=True),
        }, aggr='mean')
        self.conv2 = HeteroConv({
            ('product', 'has_brand', 'brand'): SAGEConv((-1, -1), hidden_channels, normalize=True),
            ('brand', 'rev_has_brand', 'product'): SAGEConv((-1, -1), hidden_channels, normalize=True),
            ('product', 'in_category', 'category'): SAGEConv((-1, -1), hidden_channels, normalize=True),
            ('category', 'rev_in_category', 'product'): SAGEConv((-1, -1), hidden_channels, normalize=True),
        }, aggr='mean')
        self.mlp = nn.Sequential(
            nn.Linear(hidden_channels, 64),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(32, 1),
        )

    def forward(self, data):
        x_dict = {
            'product': F.relu(self.input_proj['product'](data['product'].x)),
            'brand': F.relu(self.input_proj['brand'](self.brand_emb(data['brand'].node_id))),
            'category': F.relu(self.input_proj['category'](self.cat_emb(data['category'].node_id))),
        }
        x_dict = {key: F.dropout(value, p=self.dropout, training=self.training) for key, value in x_dict.items()}
        x_dict = self.conv1(x_dict, data.edge_index_dict)
        x_dict = {key: F.relu(value) for key, value in x_dict.items()}
        x_dict = {key: F.dropout(value, p=self.dropout, training=self.training) for key, value in x_dict.items()}
        x_dict = self.conv2(x_dict, data.edge_index_dict)
        x_dict = {key: F.relu(value) for key, value in x_dict.items()}
        x_dict = {key: F.dropout(value, p=self.dropout, training=self.training) for key, value in x_dict.items()}
        return self.mlp(x_dict['product']).squeeze(-1)


class GATRegressor(nn.Module):
    def __init__(self, in_channels_dict, num_brands, num_cats, hidden_channels=64, dropout=0.3, heads=4):
        super().__init__()
        self.hidden_channels = hidden_channels
        self.dropout = dropout
        self.heads = heads
        self.brand_emb = nn.Embedding(num_brands, hidden_channels)
        self.cat_emb = nn.Embedding(num_cats, hidden_channels)
        self.input_proj = nn.ModuleDict({
            'product': nn.Linear(in_channels_dict['product'], hidden_channels),
            'brand': nn.Linear(hidden_channels, hidden_channels),
            'category': nn.Linear(hidden_channels, hidden_channels),
        })

        self.conv1 = HeteroConv({
            ('product', 'has_brand', 'brand'): GATConv((-1, -1), hidden_channels // heads, heads=heads, dropout=dropout, add_self_loops=False),
            ('brand', 'rev_has_brand', 'product'): GATConv((-1, -1), hidden_channels // heads, heads=heads, dropout=dropout, add_self_loops=False),
            ('product', 'in_category', 'category'): GATConv((-1, -1), hidden_channels // heads, heads=heads, dropout=dropout, add_self_loops=False),
            ('category', 'rev_in_category', 'product'): GATConv((-1, -1), hidden_channels // heads, heads=heads, dropout=dropout, add_self_loops=False),
        }, aggr='mean')
        self.conv2 = HeteroConv({
            ('product', 'has_brand', 'brand'): GATConv((-1, -1), hidden_channels, heads=1, dropout=dropout, add_self_loops=False),
            ('brand', 'rev_has_brand', 'product'): GATConv((-1, -1), hidden_channels, heads=1, dropout=dropout, add_self_loops=False),
            ('product', 'in_category', 'category'): GATConv((-1, -1), hidden_channels, heads=1, dropout=dropout, add_self_loops=False),
            ('category', 'rev_in_category', 'product'): GATConv((-1, -1), hidden_channels, heads=1, dropout=dropout, add_self_loops=False),
        }, aggr='mean')
        self.mlp = nn.Sequential(
            nn.Linear(hidden_channels, 32),
            nn.ELU(),
            nn.Dropout(dropout),
            nn.Linear(32, 1),
        )

    def forward(self, data):
        x_dict = {
            'product': F.elu(self.input_proj['product'](data['product'].x)),
            'brand': F.elu(self.input_proj['brand'](self.brand_emb(data['brand'].node_id))),
            'category': F.elu(self.input_proj['category'](self.cat_emb(data['category'].node_id))),
        }
        x_dict = {key: F.dropout(value, p=self.dropout, training=self.training) for key, value in x_dict.items()}
        x_dict = self.conv1(x_dict, data.edge_index_dict)
        x_dict = {key: F.elu(value) for key, value in x_dict.items()}
        x_dict = {key: F.dropout(value, p=self.dropout, training=self.training) for key, value in x_dict.items()}
        x_dict = self.conv2(x_dict, data.edge_index_dict)
        x_dict = {key: F.elu(value) for key, value in x_dict.items()}
        x_dict = {key: F.dropout(value, p=self.dropout, training=self.training) for key, value in x_dict.items()}
        return self.mlp(x_dict['product']).squeeze(-1)


def train_gnn(model, data, epochs=60, lr=5e-3, weight_decay=1e-4, log_every=10):
    optimizer = Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    scheduler = ReduceLROnPlateau(optimizer, mode='min', patience=5, factor=0.5)
    loss_fn = nn.MSELoss()
    best_state = None
    best_val = float('inf')

    for epoch in range(1, epochs + 1):
        model.train()
        optimizer.zero_grad()
        out = model(data)
        train_loss = loss_fn(out[data['product'].train_mask], data['product'].y[data['product'].train_mask])
        train_loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        model.eval()
        with torch.no_grad():
            val_out = model(data)
            val_loss = loss_fn(val_out[data['product'].val_mask], data['product'].y[data['product'].val_mask]).item()

        scheduler.step(val_loss)

        if val_loss < best_val:
            best_val = val_loss
            best_state = {key: value.detach().cpu().clone() for key, value in model.state_dict().items()}

        if epoch == 1 or epoch % log_every == 0:
            print(f'Epoch {epoch:03d} | Train Loss: {train_loss.item():.4f} | Val Loss: {val_loss:.4f}')

    return best_state, best_val


def predict_gnn(model, data, mask):
    model.eval()
    with torch.no_grad():
        pred = model(data)
    return pred[mask].detach().cpu().numpy()


def regression_metrics(y_true_log, y_pred_log):
    y_true = np.expm1(y_true_log)
    y_pred = np.expm1(y_pred_log)
    y_pred = np.maximum(y_pred, 0)
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    rmsle = np.sqrt(mean_squared_log_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)
    return mae, rmse, rmsle, r2

## Training

GraphSAGE dan GAT dilatih pada node product dengan loss MSE pada log_price. Model terbaik disimpan berdasarkan validation loss.

In [18]:
# --- Section: Train GraphSAGE and GAT ---
sage_model = GraphSAGERegressor(in_channels_dict, num_brands, num_cats, hidden_channels=128, dropout=0.3).to(device)
best_state_sage, best_val_sage = train_gnn(sage_model, data, epochs=60)
sage_model.load_state_dict(best_state_sage)
test_pred_sage = predict_gnn(sage_model, data, data['product'].test_mask)
print('Best validation loss (GraphSAGE):', best_val_sage)

gat_model = GATRegressor(in_channels_dict, num_brands, num_cats, hidden_channels=64, dropout=0.3, heads=4).to(device)
best_state_gat, best_val_gat = train_gnn(gat_model, data, epochs=60)
gat_model.load_state_dict(best_state_gat)
test_pred_gat = predict_gnn(gat_model, data, data['product'].test_mask)
print('Best validation loss (GAT):', best_val_gat)

Epoch 001 | Train Loss: 8.7582 | Val Loss: 8.1579
Epoch 010 | Train Loss: 1.5606 | Val Loss: 0.6826
Epoch 020 | Train Loss: 0.9233 | Val Loss: 0.6596
Epoch 030 | Train Loss: 0.9114 | Val Loss: 0.5361
Epoch 040 | Train Loss: 0.8417 | Val Loss: 0.5091
Epoch 050 | Train Loss: 0.7933 | Val Loss: 0.4739
Epoch 060 | Train Loss: 0.7404 | Val Loss: 0.4192
Best validation loss (GraphSAGE): 0.419205904006958
DEBUG: GATConv(('product', 'has_brand', 'brand')).add_self_loops = False
Epoch 001 | Train Loss: 9.1908 | Val Loss: 7.6738
Epoch 010 | Train Loss: 1.9695 | Val Loss: 0.8158
Epoch 020 | Train Loss: 1.2210 | Val Loss: 0.5348
Epoch 030 | Train Loss: 0.9321 | Val Loss: 0.4867
Epoch 040 | Train Loss: 0.7411 | Val Loss: 0.4589
Epoch 050 | Train Loss: 0.6789 | Val Loss: 0.4513
Epoch 060 | Train Loss: 0.6524 | Val Loss: 0.4516
Best validation loss (GAT): 0.44767138361930847


## Evaluation and Save

Hitung metrik test set untuk semua model, kemudian simpan model dan metadata preprocessing ke disk.

In [21]:
# --- Section: Evaluate models and persist artifacts ---
y_test_log = test_df['log_price'].values
metrics = []

for model_name, pred in [
    ('TF-IDF + Ridge', test_pred_ridge),
    ('XGBoost', test_pred_xgb),
    ('GraphSAGE', test_pred_sage),
    ('GAT', test_pred_gat),
]:
    y_true = np.expm1(y_test_log)
    y_pred = np.maximum(np.expm1(pred), 0)
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    rmsle = np.sqrt(mean_squared_log_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)
    metrics.append({'Model': model_name, 'MAE': mae, 'RMSE': rmse, 'RMSLE': rmsle, 'R2': r2})

results_df = pd.DataFrame(metrics).sort_values('RMSLE').reset_index(drop=True)
print(results_df)


graphsage_path = ARTIFACT_DIR / 'graphsage_secondprice.pt'
gat_path = ARTIFACT_DIR / 'gat_secondprice.pt'
tfidf_ridge_path = ARTIFACT_DIR / 'tfidf_ridge_pipeline.pkl'
xgb_path = ARTIFACT_DIR / 'xgboost_model.pkl'
meta_path = ARTIFACT_DIR / 'preprocessor_meta.pkl'

torch.save(sage_model.state_dict(), graphsage_path)
torch.save(gat_model.state_dict(), gat_path)

with open(tfidf_ridge_path, 'wb') as f:
    pickle.dump(tfidf_ridge, f)

with open(xgb_path, 'wb') as f:
    pickle.dump(xgb_model, f)

preprocessor_meta = {
    'le_brand': le_brand,
    'le_cat_main': le_cat_main,
    'le_cat_sub1': le_cat_sub1,
    'tfidf_vec': tfidf_vec_graph,
    'in_channels_dict': in_channels_dict,
    'n_brands': num_brands,
    'n_cats': num_cats,
    'SEED': SEED,
    'columns': ['name', 'brand_name', 'category_name', 'item_condition_id', 'item_description', 'shipping'],
    'brand_mapping': brand_mapping_graph,
    'cat_main_mapping': cat_main_mapping_graph,
    'brand_unknown_index': num_brands - 1,
    'cat_unknown_index': num_cats - 1,
    'tfidf_max_features': 50000,
    'graph_feature_dim': 130,
    'max_rows': MAX_ROWS
}

with open(meta_path, 'wb') as f:
    pickle.dump(preprocessor_meta, f)

artifact_rows = []
for artifact_path in [graphsage_path, gat_path, tfidf_ridge_path, xgb_path, meta_path]:
    artifact_rows.append({
        'artifact': artifact_path.name,
        'path': str(artifact_path.resolve()),
        'size_bytes': artifact_path.stat().st_size if artifact_path.exists() else 0
    })

artifact_summary = pd.DataFrame(artifact_rows)
artifact_summary

            Model        MAE       RMSE     RMSLE        R2
0  TF-IDF + Ridge  11.778346  32.337968  0.520346  0.280087
1         XGBoost  14.233225  36.473945  0.644631  0.084159
2       GraphSAGE  14.174594  37.242196  0.646396  0.045172
3             GAT  14.637452  36.880975  0.669523  0.063605


,artifact,path,size_bytes
0,graphsage_secondprice.pt,/content/graphsage_secondprice.pt,2522009
1,gat_secondprice.pt,/content/gat_secondprice.pt,970583
2,tfidf_ridge_pipeline.pkl,/content/tfidf_ridge_pipeline.pkl,2370885
3,xgboost_model.pkl,/content/xgboost_model.pkl,2073709
4,preprocessor_meta.pkl,/content/preprocessor_meta.pkl,2320972
